# Week 2 on a GPU - embed the whole catalog in ~20 minutes

Your laptop does this at about **0.9 songs/sec**, so all 13,497 tracks take
roughly **4.2 hours** and about 7 GB of downloads over your own connection.

A free Colab T4 does the same work in about **20 minutes**, on Google's
bandwidth. The result is a small file you download and drop back into the
project - about **28 MB**, because embeddings are just numbers.

**Before you start:** turn the GPU on.
`Runtime` -> `Change runtime type` -> `Hardware accelerator` -> **T4 GPU** -> Save.

Then run the cells in order.

## 1. Check you actually got a GPU

If this says `No GPU`, go back and change the runtime type. Everything below
still works without one, it will just be as slow as your laptop.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'No GPU - change Runtime type to T4'

## 2. Install what Colab is missing

Colab already has `torch` and `transformers`. We need **PyAV** to decode the
previews - they are `.m4a` (AAC) files, not MP3, and most Python audio
libraries cannot read AAC at all.

In [ ]:
!pip install -q av faiss-cpu
import av, torch
print('PyAV', av.__version__, '| torch', torch.__version__, '| CUDA', torch.cuda.is_available())

## 3. Get the project code

In [ ]:
!git clone -q https://github.com/deepakkarhana/Music-Instagram.git
%cd Music-Instagram
!ls scripts

## 4. Upload your catalog

The catalog is **not** in the repo - `data/` is gitignored, because a repo
should ship code, not data.

Run this cell, click **Choose Files**, and pick this file from your laptop:

```
Desktop/Music-Instagram/data/raw/itunes_catalog.csv
```

It is about 4 MB, so it uploads in seconds.

**Upload the file rather than re-harvesting here.** iTunes returns slightly
different results each time, so a fresh harvest would produce a catalog whose
track ids do not match the one on your laptop - and the embeddings would then
line up with the wrong songs.

In [ ]:
import os
from google.colab import files
os.makedirs('data/raw', exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    os.replace(name, 'data/raw/itunes_catalog.csv')
print(sum(1 for _ in open('data/raw/itunes_catalog.csv', encoding='utf-8')) - 1, 'tracks loaded')

## 5. (Optional) Resume work you already did locally

If you already embedded some tracks on your laptop, upload those two files
and Colab will skip them instead of redoing the work:

```
data/interim/clap_audio.f32
data/interim/clap_audio_ids.csv
```

Skip this cell entirely if you would rather just start fresh.

In [ ]:
import os
from google.colab import files
os.makedirs('data/interim', exist_ok=True)
print('Upload clap_audio.f32 and clap_audio_ids.csv, or press Cancel to skip.')
try:
    for name in files.upload():
        os.replace(name, os.path.join('data/interim', name))
    print('resuming from existing embeddings')
except Exception:
    print('starting fresh')

## 6. Embed everything

This is the long cell - roughly 20 minutes on a T4.

Before it starts, it runs the text-encoder health check. That check exists
because we once spent an afternoon on a checkpoint whose text encoder was
silently dead. If it fails, stop - do not let it run for 20 minutes producing
meaningless numbers.

**Keep this tab open.** Colab disconnects idle sessions. If it does drop, just
rerun the cell - it resumes from exactly where it stopped.

In [ ]:
!python -u scripts/03_embed_catalog.py --batch 32 --workers 12

## 7. Build the index and check it works

In [ ]:
!python scripts/04_build_index.py

In [ ]:
!python scripts/05_search.py "rainy cafe window, quiet piano, soft melancholy" -k 5

## 8. Download the results

Two files, about 28 MB total. Put them in your local `data/interim/` folder,
then run `python scripts/04_build_index.py` on your laptop and everything
works offline from there.

We download the raw embeddings rather than the FAISS index because the index
rebuilds from them in two seconds, and these two files are the part that took
20 minutes to compute.

In [ ]:
from google.colab import files
files.download('data/interim/clap_audio.f32')
files.download('data/interim/clap_audio_ids.csv')